# einops-repeat — worked example 3: Nearest-neighbor stretch a 3D volume along the depth axis

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The `(d r)` composition on the output side of `repeat` stretches an axis so that each source slice appears `r` times consecutively — nearest-neighbor upsampling along one dimension. Used in 3D medical imaging to upsample low-resolution depth slices to match the in-plane resolution.

## Worked solution

Input: `(D=4, H=16, W=16)` MRI volume with coarse depth sampling.

**Pattern:** `'d h w -> (d r) h w'` with `r=3`.

The `(d r)` group puts `d` as the outer index (varies slower) and `r` as the inner. Depth slice `0` appears at output positions `0, 1, 2`; slice `1` at positions `3, 4, 5`; etc.

**Result shape:** `(12, 16, 16)` — 3x more depth slices, each repeated block of 3.

In [ ]:
import torch as t
from einops import repeat

t.manual_seed(33)
D, H, W = 4, 8, 8
vol = t.randn(D, H, W)

def stretch_depth(vol, r):
    return repeat(vol, 'd h w -> (d r) h w', r=r)

stretched = stretch_depth(vol, r=3)
print('Volume shape:', vol.shape)      # (4, 8, 8)
print('Stretched shape:', stretched.shape)  # (12, 8, 8)
assert stretched.shape == (D * 3, H, W)

# Each source slice d appears at output positions 3d, 3d+1, 3d+2
for d in range(D):
    for offset in range(3):
        assert t.allclose(stretched[3 * d + offset], vol[d])
print('Each depth slice repeated 3 times consecutively:', True)